# Phase 5 — GTM Recommendations

**Goal:** Translate all analysis into concrete business recommendations.

This is the most important notebook. Analysis without action is just a report no one reads.

By the end of this notebook you will know:
- How to structure a GTM recommendation
- How to prioritize recommendations by impact
- How to present findings to a business stakeholder
- How to define success metrics for each recommendation

---

## Core Concept: The Analyst's Responsibility

Your job is NOT to produce charts. Your job is to help the business make better decisions.

Every analysis must answer:
1. **What** did we find? (the number)
2. **Why does it matter?** (business impact)
3. **What should we do?** (specific action)
4. **How will we know it worked?** (metric + target)

If you cannot answer all 4 — you have not finished the analysis.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv("../data/2019-Oct.csv", nrows=500_000)
df["event_time"] = pd.to_datetime(df["event_time"])

# Recompute key metrics
views     = df[df["event_type"] == "view"]["user_id"].nunique()
carts     = df[df["event_type"] == "cart"]["user_id"].nunique()
purchases = df[df["event_type"] == "purchase"]["user_id"].nunique()

view_to_cart     = carts     / views
cart_to_purchase = purchases / carts

print("KEY METRICS RECAP")
print(f"  Views:             {views:,}")
print(f"  Carts:             {carts:,}")
print(f"  Purchases:         {purchases:,}")
print(f"  View→Cart:         {view_to_cart:.1%}")
print(f"  Cart→Purchase:     {cart_to_purchase:.1%}")

KEY METRICS RECAP
  Views:             89,108
  Carts:             4,441
  Purchases:         7,362
  View→Cart:         5.0%
  Cart→Purchase:     165.8%


## Build Your Recommendation Matrix

We will score each recommendation on:
- **Impact** — how many users does it affect? What revenue is at stake?
- **Effort** — how hard is it to implement?
- **Confidence** — how strongly does the data support this?

High Impact + Low Effort + High Confidence = DO THIS FIRST

In [2]:
# Impact calculation helpers
def impact_users(base_users, rate_improvement):
    return base_users * rate_improvement

def format_rec(number, title, finding, so_what, action, metric, impact_users, effort, confidence):
    print(f"{'='*65}")
    print(f"RECOMMENDATION {number}: {title.upper()}")
    print(f"{'─'*65}")
    print(f"FINDING    : {finding}")
    print(f"SO WHAT    : {so_what}")
    print(f"ACTION     : {action}")
    print(f"METRIC     : {metric}")
    print(f"IMPACT     : ~{impact_users:,.0f} additional users could convert")
    print(f"EFFORT     : {effort}")
    print(f"CONFIDENCE : {confidence}")
    print()

# You will fill in real numbers after running earlier notebooks
# These are template values — replace with your actual findings
v2c_gap  = max(0.05 - view_to_cart, 0)   # gap to industry floor (5%)
c2p_gap  = max(0.20 - cart_to_purchase, 0)  # gap to industry floor (20%)

format_rec(
    1,
    "Reduce Checkout Friction",
    f"Cart→Purchase rate is {cart_to_purchase:.1%} vs industry benchmark of 20-40%.",
    f"If we recover half the gap to the 20% floor, that is ~{impact_users(carts, c2p_gap/2):,.0f} additional purchases.",
    "Audit checkout flow: remove unnecessary form fields, add trust badges (SSL, guarantees), offer guest checkout.",
    "Cart→Purchase rate; target: +5pp in 60 days",
    impact_users(carts, c2p_gap / 2),
    "Low (UX changes, no backend needed)",
    "High (price analysis supports checkout friction hypothesis)"
)

format_rec(
    2,
    "Improve Product Page Engagement",
    f"View→Cart rate is {view_to_cart:.1%}. Only 1 in {1/view_to_cart:.0f} viewers adds to cart.",
    f"Improving view-to-cart by 2pp would add ~{impact_users(views, 0.02):,.0f} more cart entries.",
    "A/B test: add product videos, improve image quality, show social proof (reviews count) on product pages.",
    "View→Cart rate; target: +2pp in 30 days",
    impact_users(views, 0.02),
    "Medium (requires design + content work)",
    "Medium (drop-off data supports this but needs UX research to confirm root cause)"
)

RECOMMENDATION 1: REDUCE CHECKOUT FRICTION
─────────────────────────────────────────────────────────────────
FINDING    : Cart→Purchase rate is 165.8% vs industry benchmark of 20-40%.
SO WHAT    : If we recover half the gap to the 20% floor, that is ~0 additional purchases.
ACTION     : Audit checkout flow: remove unnecessary form fields, add trust badges (SSL, guarantees), offer guest checkout.
METRIC     : Cart→Purchase rate; target: +5pp in 60 days
IMPACT     : ~0 additional users could convert
EFFORT     : Low (UX changes, no backend needed)
CONFIDENCE : High (price analysis supports checkout friction hypothesis)

RECOMMENDATION 2: IMPROVE PRODUCT PAGE ENGAGEMENT
─────────────────────────────────────────────────────────────────
FINDING    : View→Cart rate is 5.0%. Only 1 in 20 viewers adds to cart.
SO WHAT    : Improving view-to-cart by 2pp would add ~1,782 more cart entries.
ACTION     : A/B test: add product videos, improve image quality, show social proof (reviews count) on prod

In [3]:
# Recommendation 3 — from segmentation data (fill in from your Notebook 4 findings)
print("="*65)
print("RECOMMENDATION 3: FOCUS GTM SPEND ON HIGH-CONVERTING CATEGORIES")
print("─"*65)
print("FINDING    : Category analysis shows significant variation in conversion rates.")
print("             Some categories convert 3-5x better than others.")
print("SO WHAT    : Marketing budget spent equally across categories has poor ROI.")
print("             Concentrating spend on high-converting categories improves returns.")
print("ACTION     : Reallocate 30% of paid search budget from low-converting")
print("             to top-converting categories. Test for 4 weeks.")
print("METRIC     : Revenue per marketing dollar (ROMI) by category;")
print("             target: +15% ROMI in the reallocated categories.")
print("EFFORT     : Low (budget reallocation, no product changes)")
print("CONFIDENCE : High (conversion rate difference is statistically meaningful)")
print()

print("="*65)
print("RECOMMENDATION 4: CART ABANDONMENT EMAIL CAMPAIGN")
print("─"*65)
cart_abandoned = carts - purchases
print(f"FINDING    : {cart_abandoned:,} unique users added to cart but did not purchase.")
avg_cart_price = df[df['event_type'] == 'cart']['price'].median()
potential_rev  = cart_abandoned * avg_cart_price * 0.10  # recover 10%
print(f"SO WHAT    : At median cart value ${avg_cart_price:.0f}, recovering even 10% of")
print(f"             abandoned carts represents ~${potential_rev:,.0f} in potential revenue.")
print(f"ACTION     : Implement a 3-email abandonment sequence:")
print(f"             Email 1 (1hr later): reminder with cart contents")
print(f"             Email 2 (24hr later): social proof / reviews")
print(f"             Email 3 (72hr later): limited-time discount")
print(f"METRIC     : Abandoned cart recovery rate; target: 5-8% recovery in 60 days")
print(f"EFFORT     : Medium (requires email automation setup)")
print(f"CONFIDENCE : High (industry standard, well-documented conversion lift)")

RECOMMENDATION 3: FOCUS GTM SPEND ON HIGH-CONVERTING CATEGORIES
─────────────────────────────────────────────────────────────────
FINDING    : Category analysis shows significant variation in conversion rates.
             Some categories convert 3-5x better than others.
SO WHAT    : Marketing budget spent equally across categories has poor ROI.
             Concentrating spend on high-converting categories improves returns.
ACTION     : Reallocate 30% of paid search budget from low-converting
             to top-converting categories. Test for 4 weeks.
METRIC     : Revenue per marketing dollar (ROMI) by category;
             target: +15% ROMI in the reallocated categories.
EFFORT     : Low (budget reallocation, no product changes)
CONFIDENCE : High (conversion rate difference is statistically meaningful)

RECOMMENDATION 4: CART ABANDONMENT EMAIL CAMPAIGN
─────────────────────────────────────────────────────────────────
FINDING    : -2,921 unique users added to cart but did not purcha

## Prioritization Matrix

Visualize recommendations by impact vs effort to build a roadmap.

In [4]:
fig, ax = plt.subplots(figsize=(10, 7))

recs = [
    ("Reduce Checkout\nFriction",    8.5, 3, "#7ED321"),
    ("Improve Product\nPages",       6.0, 6, "#4C9BE8"),
    ("GTM Budget\nReallocation",     7.0, 2, "#7ED321"),
    ("Cart Abandonment\nEmails",     8.0, 5, "#4C9BE8"),
]

for name, impact, effort, color in recs:
    ax.scatter(effort, impact, s=400, color=color, zorder=5, edgecolors="white", lw=2)
    ax.annotate(name, (effort, impact), textcoords="offset points",
                xytext=(10, 5), fontsize=9.5)

# Quadrant lines
ax.axhline(5, color="gray", linestyle="--", alpha=0.5)
ax.axvline(5, color="gray", linestyle="--", alpha=0.5)

ax.text(1, 9.3,  "DO FIRST
(High Impact, Low Effort)",  fontsize=9, color="#2E7D32", fontweight="bold")
ax.text(6, 9.3,  "PLAN
(High Impact, High Effort)",     fontsize=9, color="#1565C0", fontweight="bold")
ax.text(1, 1.0,  "FILL IN
(Low Impact, Low Effort)",    fontsize=9, color="gray")
ax.text(6, 1.0,  "RECONSIDER
(Low Impact, High Effort)",fontsize=9, color="#B71C1C")

ax.set_xlim(0, 11)
ax.set_ylim(0, 10)
ax.set_xlabel("Effort (1=Easy, 10=Hard)", fontsize=11)
ax.set_ylabel("Business Impact (1=Low, 10=High)", fontsize=11)
ax.set_title("GTM Recommendation Prioritization Matrix", fontsize=14, fontweight="bold")

green_patch = mpatches.Patch(color="#7ED321", label="Do First")
blue_patch  = mpatches.Patch(color="#4C9BE8", label="Plan & Schedule")
ax.legend(handles=[green_patch, blue_patch], loc="lower right")

plt.tight_layout()
plt.savefig("../outputs/05_recommendation_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

SyntaxError: unterminated string literal (detected at line 19) (3098354501.py, line 19)

## Executive Summary (1-Page Format)

This is what you would present to a VP or CEO. Clear, data-backed, decision-ready.

In [5]:
summary = (
    "GTM ANALYSIS - EXECUTIVE SUMMARY\n"
    "E-Commerce Behavior Data, Oct 2019\n"
    "=" * 60 + "\n\n"
    "KEY FINDING\n"
    "  We analyzed 500,000 user events across the full purchase\n"
    "  funnel. Biggest opportunity: checkout conversion is below benchmark.\n\n"
    "FUNNEL PERFORMANCE\n"
    "  Stage          Users         Rate       Benchmark\n"
    "  View           [from data]   --         --\n"
    "  Add to Cart    [from data]   [rate]     5-15%\n"
    "  Purchase       [from data]   [rate]     20-40%\n\n"
    "TOP 3 RECOMMENDATIONS\n\n"
    "  1. REDUCE CHECKOUT FRICTION  [High Impact / Low Effort]\n"
    "     Audit checkout; add trust signals\n"
    "     Target: +5pp cart-to-purchase in 60 days\n\n"
    "  2. CART ABANDONMENT EMAILS   [High Impact / Med Effort]\n"
    "     3-email sequence for cart abandoners\n"
    "     Target: 5-8% cart recovery rate\n\n"
    "  3. REALLOCATE GTM BUDGET     [High Impact / Low Effort]\n"
    "     Shift 30% spend to highest-converting categories\n"
    "     Target: +15% ROMI in reallocated categories\n"
    + "=" * 60
)
print(summary)

GTM ANALYSIS - EXECUTIVE SUMMARY
E-Commerce Behavior Data, Oct 2019
=GTM ANALYSIS - EXECUTIVE SUMMARY
E-Commerce Behavior Data, Oct 2019
=GTM ANALYSIS - EXECUTIVE SUMMARY
E-Commerce Behavior Data, Oct 2019
=GTM ANALYSIS - EXECUTIVE SUMMARY
E-Commerce Behavior Data, Oct 2019
=GTM ANALYSIS - EXECUTIVE SUMMARY
E-Commerce Behavior Data, Oct 2019
=GTM ANALYSIS - EXECUTIVE SUMMARY
E-Commerce Behavior Data, Oct 2019
=GTM ANALYSIS - EXECUTIVE SUMMARY
E-Commerce Behavior Data, Oct 2019
=GTM ANALYSIS - EXECUTIVE SUMMARY
E-Commerce Behavior Data, Oct 2019
=GTM ANALYSIS - EXECUTIVE SUMMARY
E-Commerce Behavior Data, Oct 2019
=GTM ANALYSIS - EXECUTIVE SUMMARY
E-Commerce Behavior Data, Oct 2019
=GTM ANALYSIS - EXECUTIVE SUMMARY
E-Commerce Behavior Data, Oct 2019
=GTM ANALYSIS - EXECUTIVE SUMMARY
E-Commerce Behavior Data, Oct 2019
=GTM ANALYSIS - EXECUTIVE SUMMARY
E-Commerce Behavior Data, Oct 2019
=GTM ANALYSIS - EXECUTIVE SUMMARY
E-Commerce Behavior Data, Oct 2019
=GTM ANALYSIS - EXECUTIVE SUMMARY
E

## Final Checkpoint — The 5 Questions Every GTM Analyst Must Answer

After completing all 5 notebooks, you should be able to answer these from memory:

1. What is the end-to-end (view-to-purchase) conversion rate for this store?
2. At which stage is the biggest drop-off?
3. What is one data-backed hypothesis for why that drop-off happens?
4. Which segment (by price tier or category) should the business prioritize?
5. Write one recommendation in the format: FINDING → SO WHAT → ACTION → METRIC

**If you can answer all 5 clearly and confidently, you have done GTM analysis.**

---

## What You Have Built

By completing this project you have:

- ✅ Loaded and explored a real-world e-commerce dataset
- ✅ Built a full top-to-bottom funnel analysis
- ✅ Identified where and why users drop off
- ✅ Segmented users by price, brand, day, and spending behavior
- ✅ Produced prioritized, data-backed GTM recommendations
- ✅ Created charts suitable for a business presentation

This is the exact skill set used by GTM analysts, growth analysts, and product analysts at tech companies.

---
**You are done. Now go back and fill in all the `[from data]` placeholders in the executive summary with your actual numbers.**